In [13]:
# ==============================================================
# B — Dual Head Pipeline
# (1) Scope-detector: ALL vs NOT_ALL
# (2) Time-detector: HOUR/DAY/WEEK/MONTH/YEAR/ALL
# Model: IndoBERT (bisa ganti ke "roberta-base")
# ==============================================================

import os, json, csv, random
from collections import Counter
import numpy as np
import torch
from datasets import Dataset, DatasetDict, load_dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer, EarlyStoppingCallback)
from sklearn.metrics import accuracy_score, f1_score, classification_report

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

# ---------- PATH INPUT (ganti kalau beda) ----------
TRAIN_B = "/workspace/train_trend_B_bindex.jsonl"
VALID_B = "/workspace/valid_trend_B_bindex.jsonl"
LABELS_B = "/workspace/labels_trend_B.csv"   # kolom: code,scope,granularity

# ---------- PATH OUTPUT ----------
OUT_ROOT = "./runs_B_dual"
os.makedirs(OUT_ROOT, exist_ok=True)

# ---------- BASE MODEL ----------
BASE_MODEL = "indobenchmark/indobert-base-p1"   # bisa ganti "roberta-base"

# ---------- 0) Utility ----------
def load_jsonl(p):
    with open(p, "r", encoding="utf-8") as f:
        return [json.loads(x) for x in f]

def read_label_index(csv_path):
    with open(csv_path, newline="", encoding="utf-8") as f:
        # deteksi delimiter ; atau ,
        sample = f.read(2048)
        f.seek(0)
        try:
            dialect = csv.Sniffer().sniff(sample, delimiters=";,")
        except Exception:
            class _D: delimiter = ';'
            dialect = _D()

        reader = csv.DictReader(f, dialect=dialect)
        idx = {}
        for row in reader:
            # normalisasi key
            row = { (k or "").strip().lower(): (v or "").strip() for k,v in row.items() }
            code = row.get("code") or row.get("kode")
            scope = row.get("scope")
            gran  = row.get("granularity") or row.get("granularitas")
            if not (code and scope and gran):
                raise ValueError(f"Baris CSV tidak lengkap: {row}")
            idx[code] = {"scope": scope.upper(), "granularity": gran.upper()}
        return idx

# ---------- 1) Build two derived datasets ----------
# Dari dataset B (b1..b18):
# - Scope: "ALL" jika scope=ALL, selain itu "NOT_ALL"
# - Time : granularity langsung jadi label (HOUR/DAY/WEEK/MONTH/YEAR/ALL)

def build_dual_datasets(train_path, valid_path, label_csv, out_dir="./data_B_dual"):
    os.makedirs(out_dir, exist_ok=True)
    idx = read_label_index(label_csv)

    def convert_split(rows):
        scope_rows, time_rows = [], []
        for r in rows:
            code = r["label"]             # ex: b7
            meta = idx[code]              # scope, granularity
            scope_label = "ALL" if meta["scope"] == "ALL" else "NOT_ALL"
            time_label = meta["granularity"]  # HOUR/DAY/WEEK/MONTH/YEAR/ALL
            scope_rows.append({"text": r["text"], "label": scope_label})
            time_rows.append({"text": r["text"], "label": time_label})
        return scope_rows, time_rows

    tr_all = load_jsonl(train_path)  # [{"text":..., "label": "b#"}]
    va_all = load_jsonl(valid_path)

    tr_scope, tr_time = convert_split(tr_all)
    va_scope, va_time = convert_split(va_all)

    # Simpan
    fp = {}
    fp["train_scope"] = os.path.join(out_dir, "train_scope.jsonl")
    fp["valid_scope"] = os.path.join(out_dir, "valid_scope.jsonl")
    fp["train_time"]  = os.path.join(out_dir, "train_time.jsonl")
    fp["valid_time"]  = os.path.join(out_dir, "valid_time.jsonl")

    with open(fp["train_scope"], "w", encoding="utf-8") as f:
        for x in tr_scope: f.write(json.dumps(x, ensure_ascii=False)+"\n")
    with open(fp["valid_scope"], "w", encoding="utf-8") as f:
        for x in va_scope: f.write(json.dumps(x, ensure_ascii=False)+"\n")
    with open(fp["train_time"], "w", encoding="utf-8") as f:
        for x in tr_time: f.write(json.dumps(x, ensure_ascii=False)+"\n")
    with open(fp["valid_time"], "w", encoding="utf-8") as f:
        for x in va_time: f.write(json.dumps(x, ensure_ascii=False)+"\n")

    return fp

# ---------- 2) Training helpers ----------
def compute_metrics_mc(p):
    logits, y = p
    preds = logits.argmax(axis=1)
    return {
        "acc": accuracy_score(y, preds),
        "f1_macro": f1_score(y, preds, average="macro")
    }

def train_multiclass_from_jsonl(model_name, train_jsonl, valid_jsonl, out_dir, epochs=6, lr=3e-5, batch=16):
    # Load dataset HF dari file JSONL
    ds = load_dataset("json", data_files={"train": train_jsonl, "validation": valid_jsonl})
    labels = sorted(list({ex["label"] for ex in ds["train"]}))
    label2id = {l:i for i,l in enumerate(labels)}
    id2label = {i:l for l,i in label2id.items()}

    tok = AutoTokenizer.from_pretrained(model_name)

    def _pp(batch):
        enc = tok(batch["text"], truncation=True, padding="max_length", max_length=128)
        enc["labels"] = [int(label2id[y]) for y in batch["label"]]
        return enc

    enc = ds.map(_pp, batched=True, remove_columns=["text","label"])
    cols = ["input_ids","attention_mask","labels"]
    if "token_type_ids" in enc["train"].features: cols.append("token_type_ids")
    enc.set_format(type="torch", columns=cols)

    model = AutoModelForSequenceClassification.from_pretrained(
        model_name, num_labels=len(labels), id2label=id2label, label2id=label2id
    )

    # Untuk menghindari error penyimpanan optimizer, kita simpan manual saja
    args = TrainingArguments(
        output_dir=out_dir,
        eval_strategy="epoch",
        save_strategy="no",            # ← no autosave; kita save manual di akhir
        load_best_model_at_end=False,
        learning_rate=lr,
        num_train_epochs=epochs,
        per_device_train_batch_size=batch,
        metric_for_best_model="eval_f1_macro",
        per_device_eval_batch_size=batch*2,
        weight_decay=0.01,
        logging_steps=50,
        report_to="none",
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=enc["train"],
        eval_dataset=enc["validation"],
        tokenizer=tok,
        compute_metrics=compute_metrics_mc,
       
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
    )

    trainer.train()
    eval_res = trainer.evaluate()

    # Simpan model + tokenizer manual, aman & ringkas
    save_dir = os.path.join(out_dir, "final_model")
    os.makedirs(save_dir, exist_ok=True)
    trainer.save_model(save_dir)
    tok.save_pretrained(save_dir)

    # Simpan classification report
    pred_out = trainer.predict(enc["validation"])
    preds = np.argmax(pred_out.predictions, axis=1)
    y_true = pred_out.label_ids
    from sklearn.metrics import classification_report
    rep = classification_report(y_true, preds, target_names=[id2label[i] for i in range(len(id2label))], digits=4)
    with open(os.path.join(out_dir, "classification_report.txt"), "w") as f:
        f.write(rep)

    # Simpan peta label
    with open(os.path.join(out_dir, "label_map.json"), "w") as f:
        json.dump({"id2label": id2label, "label2id": label2id}, f, ensure_ascii=False, indent=2)

    return save_dir, eval_res, id2label, label2id

# ---------- 3) Build datasets, Train both models ----------
paths = build_dual_datasets(TRAIN_B, VALID_B, LABELS_B, out_dir="./data_B_dual")

# (A) SCOPE: ALL vs NOT_ALL
scope_dir, scope_eval, scope_i2l, scope_l2i = train_multiclass_from_jsonl(
    BASE_MODEL, paths["train_scope"], paths["valid_scope"], out_dir=os.path.join(OUT_ROOT, "scope"), epochs=6
)

# (B) TIME: HOUR/DAY/WEEK/MONTH/YEAR/ALL
time_dir, time_eval, time_i2l, time_l2i = train_multiclass_from_jsonl(
    BASE_MODEL, paths["train_time"], paths["valid_time"], out_dir=os.path.join(OUT_ROOT, "time"), epochs=6
)

print("\n== Eval Scope ==", scope_eval)
print("== Eval Time  ==", time_eval)

# ---------- 4) Runner: jalankan kedua model ----------
from transformers import AutoConfig

class DualRunner:
    def __init__(self, scope_model_dir, time_model_dir):
        # Scope
        self.scope_tok = AutoTokenizer.from_pretrained(scope_model_dir)
        self.scope_cfg = AutoConfig.from_pretrained(scope_model_dir)
        self.scope_i2l = self.scope_cfg.id2label
        self.scope_m = AutoModelForSequenceClassification.from_pretrained(scope_model_dir).eval()

        # Time
        self.time_tok = AutoTokenizer.from_pretrained(time_model_dir)
        self.time_cfg = AutoConfig.from_pretrained(time_model_dir)
        self.time_i2l = self.time_cfg.id2label
        self.time_m = AutoModelForSequenceClassification.from_pretrained(time_model_dir).eval()

        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.scope_m.to(self.device); self.time_m.to(self.device)

    @staticmethod
    def _decode(i2l, idx: int):
            # i2l bisa dict (key int/str) atau list
            if isinstance(i2l, dict):
                return i2l.get(idx) or i2l.get(str(idx)) or i2l.get(int(idx)) or f"label_{idx}"
            # kalau list/tuple
            try:
                return i2l[idx]
            except Exception:
                return f"label_{idx}"
        
    @torch.no_grad()
    def predict(self, text: str):
        # Scope
        s = self.scope_tok(text, truncation=True, padding=True, max_length=128, return_tensors="pt").to(self.device)
        s_logits = self.scope_m(**s).logits.squeeze(0).cpu().numpy()
        s_id = int(s_logits.argmax())
        s_label = self._decode(self.scope_i2l, s_id)
        s_conf = float(torch.softmax(torch.from_numpy(s_logits), dim=0)[s_id])

        # Time
        t = self.time_tok(text, truncation=True, padding=True, max_length=128, return_tensors="pt").to(self.device)
        t_logits = self.time_m(**t).logits.squeeze(0).cpu().numpy()
        t_id = int(t_logits.argmax())
        t_label = self._decode(self.time_i2l, t_id)
        t_conf = float(torch.softmax(torch.from_numpy(t_logits), dim=0)[t_id])

        return {
            "scope": {"label": s_label, "confidence": s_conf},
            "time":  {"label": t_label, "confidence": t_conf},
        }

# Contoh pakai runner
runner = DualRunner(scope_dir, time_dir)

examples = [
    "Tolong tampilkan ringkasan semua mesin dari november sampai desember tahun lalu ini",
    "Cek tren mesin XP888A hari ini",
    "Apakah ada fluktuasi besar pada mesin XP888A dan PQ667 bulan ini?",
    "Mohon tampilkan nilai tertinggi semua mesin tahun lalu"
]
for q in examples:
    print("\nQ:", q)
    print(runner.predict(q))


Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/1080 [00:00<?, ? examples/s]

Map:   0%|          | 0/216 [00:00<?, ? examples/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at indobenchmark/indobert-base-p1 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipykernel_5320/3199327112.py:153: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
Using EarlyStoppingCallback without load_best_model_at_end=True. Once training is finished, the best model will not be loaded automatically.


...

Map:   0%|          | 0/1080 [00:00<?, ? examples/s]

Map:   0%|          | 0/216 [00:00<?, ? examples/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at indobenchmark/indobert-base-p1 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipykernel_5320/3199327112.py:153: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
Using EarlyStoppingCallback without load_best_model_at_end=True. Once training is finished, the best model will not be loaded automatically.


Epoch,Training Loss,Validation Loss,Acc,F1 Macro
1,0.772100,0.004782,1.000000,1.000000
2,0.006000,0.002325,1.000000,1.000000
3,0.002400,0.001660,1.000000,1.000000



== Eval Scope == {'eval_loss': 5.438161679194309e-05, 'eval_acc': 1.0, 'eval_f1_macro': 1.0, 'eval_runtime': 0.4935, 'eval_samples_per_second': 437.679, 'eval_steps_per_second': 14.184, 'epoch': 3.0}
== Eval Time  == {'eval_loss': 0.0016602628165856004, 'eval_acc': 1.0, 'eval_f1_macro': 1.0, 'eval_runtime': 0.5143, 'eval_samples_per_second': 420.012, 'eval_steps_per_second': 13.612, 'epoch': 3.0}

Q: Tolong tampilkan ringkasan semua mesin dari november sampai desember tahun lalu ini
{'scope': {'label': 'ALL', 'confidence': 0.9999270439147949}, 'time': {'label': 'YEAR', 'confidence': 0.9981776475906372}}

Q: Cek tren mesin XP888A hari ini
{'scope': {'label': 'NOT_ALL', 'confidence': 0.9999533891677856}, 'time': {'label': 'DAY', 'confidence': 0.9982629418373108}}

Q: Apakah ada fluktuasi besar pada mesin XP888A dan PQ667 bulan ini?
{'scope': {'label': 'NOT_ALL', 'confidence': 0.9999545812606812}, 'time': {'label': 'MONTH', 'confidence': 0.9985180497169495}}

Q: Mohon tampilkan nilai ter

In [20]:
# ============================================================
# HOTFIX: Enrich labels_trend_C_index.csv (code,name,desc)
# -> add columns: scope, granularity, case
# -> write to: /mnt/data/labels_trend_C_index.enriched.csv
# -> build_three_datasets() ulang dengan CSV baru
# ============================================================

import os, csv, re, json
from pathlib import Path


RAW_CSV  = "/workspace/labels_trend_C_index.csv"  # file asli (minimal: code,name,desc)
OUT_CSV  = "/workspace/labels_trend_C_index.enriched.csv"  # file enriched (lengkap)
TRAIN_JSONL = "/workspace/train_trend_C_complex.jsonl" 
OUT_DIR = "./runs_C_triple/data_C_triple_hotfix"
SEED = 42

assert Path(RAW_CSV).exists(), f"CSV tidak ditemukan: {RAW_CSV}"
assert Path(TRAIN_JSONL).exists(), f"JSONL tidak ditemukan: {TRAIN_JSONL}"

def _tokens(s: str):
    s = (s or "").upper()
    # pisah underscore & non-alfanumerik -> token
    s = re.sub(r"[^A-Z0-9]+", " ", s)
    return set(s.split())

def infer_scope(name: str, desc: str):
    T = _tokens(name) | _tokens(desc)
    # Heuristik: kalau mengandung ALL/SEMUA => ALL, else NOT_ALL
    if "ALL" in T or "SEMUA" in T or "ALLMACHINES" in T:
        return "ALL"
    return "NOT_ALL"

def infer_granularity(name: str, desc: str):
    T = _tokens(name) | _tokens(desc)
    for g in ["HOUR", "DAY", "WEEK", "MONTH", "YEAR", "ALL"]:
        if g in T:
            return g
    # fallback kecil: lihat kata kunci bahasa Indo
    if any(k in T for k in ["JAM"]): return "HOUR"
    if any(k in T for k in ["HARI", "HARIAN", "TODAY", "KEMARIN"]): return "DAY"
    if "MINGGU" in T or "PEKAN" in T: return "WEEK"
    if "BULAN" in T or "BULANAN" in T: return "MONTH"
    if "TAHUN" in T or "TAHUNAN" in T: return "YEAR"
    # terakhir banget
    return "ALL"

REL_MARKERS = {"LASTYEAR","THISYEAR","LASTMONTH","THISMONTH","TODAY","KEMARIN","YESTERDAY",
               "TAHUN","BULAN","MINGGU","HARI","LALU","INI"}  # deteksi relatif/kalimat Indo

def infer_case(name: str, desc: str):
    T = _tokens(name) | _tokens(desc)
    # Case yang jelas:
    if "COMPARE" in T or "BANDING" in T or "PERBANDINGAN" in T:
        return "COMPARE"
    if "ALLTIME" in T:
        return "ALL_TIME"
    if "SUMMARY" in T or "RINGKAS" in T or "RANGKUM" in T:
        return "SUMMARY"
    if "SINGLE_POINT" in T or "POINT" in T:
        return "SINGLE_POINT"
    if "SINGLE_PERIOD" in T:
        return "SINGLE_PERIOD"
    # RANGE logic:
    if "RANGE" in T or "RENTANG" in T:
        # kalau ada penanda relatif (tahun lalu, bulan ini, dst.)
        if (T & REL_MARKERS):
            return "RANGE_REL"
        return "RANGE_ABS"
    # Jika mengandung satuan waktu tapi tanpa RANGE => periode tunggal
    if any(u in T for u in ["HOUR","DAY","WEEK","MONTH","YEAR","JAM","HARI","MINGGU","BULAN","TAHUN"]):
        return "SINGLE_PERIOD"
    # fallback paling aman
    return "SUMMARY"

# Baca CSV ringan dan tulis versi lengkap
with open(RAW_CSV, "r", encoding="utf-8") as f_in, \
     open(OUT_CSV, "w", encoding="utf-8", newline="") as f_out:
    reader = csv.DictReader(f_in)
    # normalisasi header agar robust
    fieldmap = { (k or "").strip().lower(): k for k in reader.fieldnames }
    code_key = fieldmap.get("code") or fieldmap.get("kode")
    name_key = fieldmap.get("name") or fieldmap.get("label") or fieldmap.get("nama")
    desc_key = fieldmap.get("desc") or fieldmap.get("description") or fieldmap.get("keterangan")

    assert code_key and name_key, f"Kolom wajib 'code' dan 'name' tidak ditemukan. Header: {reader.fieldnames}"

    writer = csv.DictWriter(f_out, fieldnames=["code","name","desc","scope","granularity","case"], delimiter=";")
    writer.writeheader()

    rows_out = 0
    for row in reader:
        code = (row.get(code_key) or "").strip()
        name = (row.get(name_key) or "").strip()
        desc = (row.get(desc_key) or "").strip()

        if not code or not name:
            # skip baris kosong
            continue

        scope = infer_scope(name, desc)
        gran  = infer_granularity(name, desc)
        case  = infer_case(name, desc)

        writer.writerow({
            "code": code,
            "name": name,
            "desc": desc,
            "scope": scope,
            "granularity": gran,
            "case": case
        })
        rows_out += 1

print(f"Enriched CSV ditulis ke: {OUT_CSV} (total {rows_out} baris)")

# ===== Re-define helper minimal untuk build dataset dari enriched CSV =====
import json
from sklearn.model_selection import StratifiedShuffleSplit
import numpy as np

def load_jsonl(p):
    with open(p, "r", encoding="utf-8") as f:
        return [json.loads(x) for x in f]

def stratify_split(rows, test_size=0.12, seed=SEED):
    y = [r["label"] for r in rows]
    idxs = np.arange(len(rows))
    sss = StratifiedShuffleSplit(n_splits=1, test_size=test_size, random_state=seed)
    tr_idx, va_idx = next(sss.split(idxs, y))
    return [rows[i] for i in tr_idx], [rows[i] for i in va_idx]

def read_label_index_enriched(csv_path: str):
    idx = {}
    with open(csv_path, "r", encoding="utf-8") as f:
        reader = csv.DictReader(f, delimiter=";")
        for row in reader:
            code = (row["code"] or "").strip()
            scope = (row["scope"] or "").strip().upper()
            gran  = (row["granularity"] or "").strip().upper()
            case  = (row["case"] or "").strip().upper()
            if code:
                idx[code] = {"scope": scope, "granularity": gran, "case": case}
    return idx

def save_jsonl(path, rows):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        for x in rows:
            f.write(json.dumps(x, ensure_ascii=False) + "\n")

def build_three_datasets_from_enriched(train_jsonl: str, enriched_csv: str, out_dir: str):
    os.makedirs(out_dir, exist_ok=True)
    idx = read_label_index_enriched(enriched_csv)

    all_rows = load_jsonl(train_jsonl)
    tr_rows, va_rows = stratify_split(all_rows, test_size=0.12, seed=SEED)

    def convert(rows):
        scope_rows, time_rows, case_rows = [], [], []
        for r in rows:
            code = r["label"]
            meta = idx.get(code)
            if not meta:
                raise KeyError(f"Kode '{code}' tidak ada di enriched CSV.")
            scope_label = "ALL" if meta["scope"] == "ALL" else "NOT_ALL"
            time_label  = meta["granularity"]
            case_label  = meta["case"]
            scope_rows.append({"text": r["text"], "label": scope_label})
            time_rows.append({"text": r["text"], "label": time_label})
            case_rows.append({"text": r["text"], "label": case_label})
        return scope_rows, time_rows, case_rows

    tr_scope, tr_time, tr_case = convert(tr_rows)
    va_scope, va_time, va_case = convert(va_rows)

    fp = {
        "train_scope": os.path.join(out_dir, "train_scope.jsonl"),
        "valid_scope": os.path.join(out_dir, "valid_scope.jsonl"),
        "train_time" : os.path.join(out_dir, "train_time.jsonl"),
        "valid_time" : os.path.join(out_dir, "valid_time.jsonl"),
        "train_case" : os.path.join(out_dir, "train_case.jsonl"),
        "valid_case" : os.path.join(out_dir, "valid_case.jsonl"),
    }
    save_jsonl(fp["train_scope"], tr_scope); save_jsonl(fp["valid_scope"], va_scope)
    save_jsonl(fp["train_time"],  tr_time);  save_jsonl(fp["valid_time"],  va_time)
    save_jsonl(fp["train_case"],  tr_case);  save_jsonl(fp["valid_case"],  va_case)
    return fp

paths = build_three_datasets_from_enriched(TRAIN_JSONL, OUT_CSV, OUT_DIR)

print("\n=== RINGKASAN DATASETS TURUNAN (HOTFIX) ===")
for k,v in paths.items():
    print(f"{k}: {v}")


Enriched CSV ditulis ke: /workspace/labels_trend_C_index.enriched.csv (total 12 baris)

=== RINGKASAN DATASETS TURUNAN (HOTFIX) ===
train_scope: ./runs_C_triple/data_C_triple_hotfix/train_scope.jsonl
valid_scope: ./runs_C_triple/data_C_triple_hotfix/valid_scope.jsonl
train_time: ./runs_C_triple/data_C_triple_hotfix/train_time.jsonl
valid_time: ./runs_C_triple/data_C_triple_hotfix/valid_time.jsonl
train_case: ./runs_C_triple/data_C_triple_hotfix/train_case.jsonl
valid_case: ./runs_C_triple/data_C_triple_hotfix/valid_case.jsonl


In [21]:
# =============================================================
# Pipeline C — Triple Head (Scope • Time • Case)
# End-to-end: CONFIG -> DATA BUILD -> TRAIN (3 heads) -> RUNNER -> DEMO
# =============================================================

# ====== (0) CONFIG ======
TRAIN_JSONL = "/workspace/train_trend_C_complex.jsonl"   # ganti jika perlu
LABEL_CSV   = "/workspace/labels_trend_C_index.enriched.csv"      # ganti jika perlu
OUT_ROOT    = "./runs_C_triple"                         # folder output
BASE_MODEL  = "indobenchmark/indobert-base-p1"          # opsi lain: "roberta-base"

EPOCHS  = 6
LR      = 3e-5
BATCH   = 16
MAX_LEN = 128
SEED    = 42

# ====== (1) IMPORTS & SETUP ======
import os, json, csv, random, itertools, math, sys, subprocess
from pathlib import Path
from typing import Dict, List, Any, Tuple
import numpy as np
random.seed(SEED); np.random.seed(SEED)

print("=== Cek file input ===")
print(" - JSONL:", Path(TRAIN_JSONL).exists(), TRAIN_JSONL)
print(" - CSV  :", Path(LABEL_CSV).exists(), LABEL_CSV)

# ====== (2) (Opsional) Install deps jika perlu ======
# Catatan: Uncomment baris di bawah jika paket belum tersedia (atau jalankan manual).
# !pip install -q transformers datasets accelerate scikit-learn

# ====== (3) UTILITIES ======
def load_jsonl(p: str) -> List[Dict[str, Any]]:
    with open(p, "r", encoding="utf-8") as f:
        return [json.loads(x) for x in f]

def detect_delimiter_from_file(csv_path: str) -> str:
    with open(csv_path, "r", encoding="utf-8") as f:
        sample = f.read(2048)
        f.seek(0)
        try:
            return csv.Sniffer().sniff(sample, delimiters=";,").delimiter
        except Exception:
            return ";"

def read_label_index_c(csv_path: str) -> Dict[str, Dict[str, str]]:
    """
    Kolom fleksibel (case-insensitive):
      - code/kode
      - scope
      - granularity/granularitas
      - case/kasus
    Return: { code: {"scope":..., "granularity":..., "case":...} } (UPPER)
    """
    delim = detect_delimiter_from_file(csv_path)
    idx = {}
    with open(csv_path, "r", encoding="utf-8") as f:
        reader = csv.DictReader(f, delimiter=delim)
        for row in reader:
            row = {(k or "").strip().lower(): (v or "").strip() for k, v in row.items()}
            code = row.get("code") or row.get("kode")
            scope = row.get("scope")
            gran  = row.get("granularity") or row.get("granularitas")
            kasus = row.get("case") or row.get("kasus")
            if not (code and scope and gran and kasus):
                raise ValueError(f"Baris CSV tidak lengkap: {row}")
            idx[code] = {"scope": scope.upper(),
                         "granularity": gran.upper(),
                         "case": kasus.upper()}
    return idx

from sklearn.model_selection import StratifiedShuffleSplit
def stratify_split(rows: List[Dict[str, Any]], test_size=0.12, seed=SEED) -> Tuple[List[Dict[str, Any]], List[Dict[str, Any]]]:
    y = [r["label"] for r in rows]
    idxs = np.arange(len(rows))
    sss = StratifiedShuffleSplit(n_splits=1, test_size=test_size, random_state=seed)
    tr_idx, va_idx = next(sss.split(idxs, y))
    tr = [rows[i] for i in tr_idx]
    va = [rows[i] for i in va_idx]
    return tr, va

def save_jsonl(path: str, rows: List[Dict[str, Any]]):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        for x in rows:
            f.write(json.dumps(x, ensure_ascii=False) + "\n")

def build_three_datasets(train_jsonl: str, label_csv: str, out_dir:str, valid_jsonl: str = "", auto_split=True) -> Dict[str, str]:
    os.makedirs(out_dir, exist_ok=True)
    idx = read_label_index_c(label_csv)

    def convert(rows):
        scope_rows, time_rows, case_rows = [], [], []
        for r in rows:
            code = r["label"]
            if code not in idx:
                raise KeyError(f"Kode '{code}' tidak ditemukan di CSV index.")
            meta = idx[code]
            scope_label = "ALL" if meta["scope"] == "ALL" else "NOT_ALL"
            time_label  = meta["granularity"]   # HOUR/DAY/WEEK/MONTH/YEAR/ALL
            case_label  = meta["case"]          # mis. SINGLE_PERIOD, RANGE_ABS, RANGE_REL, ALL_TIME, COMPARE, SUMMARY
            scope_rows.append({"text": r["text"], "label": scope_label})
            time_rows.append({"text": r["text"], "label": time_label})
            case_rows.append({"text": r["text"], "label": case_label})
        return scope_rows, time_rows, case_rows

    all_rows = load_jsonl(train_jsonl)
    if valid_jsonl and Path(valid_jsonl).exists():
        tr_rows = all_rows
        va_rows = load_jsonl(valid_jsonl)
    elif auto_split:
        tr_rows, va_rows = stratify_split(all_rows, test_size=0.12, seed=SEED)
    else:
        tr_rows, va_rows = all_rows, []

    tr_scope, tr_time, tr_case = convert(tr_rows)
    va_scope, va_time, va_case = convert(va_rows) if va_rows else ([], [], [])

    fp = {
        "train_scope": os.path.join(out_dir, "train_scope.jsonl"),
        "valid_scope": os.path.join(out_dir, "valid_scope.jsonl"),
        "train_time" : os.path.join(out_dir, "train_time.jsonl"),
        "valid_time" : os.path.join(out_dir, "valid_time.jsonl"),
        "train_case" : os.path.join(out_dir, "train_case.jsonl"),
        "valid_case" : os.path.join(out_dir, "valid_case.jsonl"),
    }
    save_jsonl(fp["train_scope"], tr_scope); save_jsonl(fp["valid_scope"], va_scope)
    save_jsonl(fp["train_time"],  tr_time);  save_jsonl(fp["valid_time"],  va_time)
    save_jsonl(fp["train_case"],  tr_case);  save_jsonl(fp["valid_case"],  va_case)
    print(">> Tersimpan dataset turunan di:", out_dir)
    return fp

print("\n=== Preview data (3 baris JSONL & 6 baris CSV) ===")
if Path(TRAIN_JSONL).exists():
    with open(TRAIN_JSONL, "r", encoding="utf-8") as f:
        for i, line in zip(range(3), f):
            print("[JSONL]", line.strip())
if Path(LABEL_CSV).exists():
    with open(LABEL_CSV, "r", encoding="utf-8") as f:
        sample = f.read(2048); f.seek(0)
        try:
            dialect = csv.Sniffer().sniff(sample, delimiters=";,")
        except Exception:
            class _D: delimiter = ';'
            dialect = _D()
        reader = csv.reader(f, delimiter=dialect.delimiter)
        for i, row in zip(range(6), reader):
            print("[CSV]", row)

# ====== (4) BUILD THREE DATASETS ======
paths = build_three_datasets(
    TRAIN_JSONL, LABEL_CSV,
    out_dir=f"{OUT_ROOT}/data_C_triple",
    valid_jsonl="", auto_split=True
)
print("paths:", paths)

# ====== (5) TRAIN HELPERS & TRAINING 3 HEADS ======
from datasets import load_dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer, EarlyStoppingCallback)
from sklearn.metrics import accuracy_score, f1_score, classification_report
import torch

def compute_metrics_mc(p):
    logits, y = p
    preds = logits.argmax(axis=1)
    return {"acc": accuracy_score(y, preds), "f1_macro": f1_score(y, preds, average="macro")}

def train_multiclass_from_jsonl(model_name, train_jsonl, valid_jsonl, out_dir,
                                epochs=6, lr=3e-5, batch=16, max_len=128):
    ds = load_dataset("json", data_files={"train": train_jsonl, "validation": valid_jsonl})
    labels = sorted(list({ex["label"] for ex in ds["train"]}))
    label2id = {l:i for i,l in enumerate(labels)}
    id2label = {i:l for i,l in enumerate(labels)}

    tok = AutoTokenizer.from_pretrained(model_name)

    def _pp(batch):
        enc = tok(batch["text"], truncation=True, padding="max_length", max_length=max_len)
        enc["labels"] = [label2id[y] for y in batch["label"]]
        return enc

    enc = ds.map(_pp, batched=True, remove_columns=["text","label"])
    cols = ["input_ids","attention_mask","labels"]
    if "token_type_ids" in enc["train"].features: cols.append("token_type_ids")
    enc.set_format(type="torch", columns=cols)

    model = AutoModelForSequenceClassification.from_pretrained(
        model_name, num_labels=len(labels), id2label=id2label, label2id=label2id
    )

    args = TrainingArguments(
        output_dir=out_dir,
        eval_strategy="epoch",
        save_strategy="no",
        load_best_model_at_end=False,
        learning_rate=lr,
        num_train_epochs=epochs,
        per_device_train_batch_size=batch,
        per_device_eval_batch_size=batch*2,
        weight_decay=0.01,
        logging_steps=50,
        report_to="none",
        metric_for_best_model="eval_f1_macro",
        seed=SEED,
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=enc["train"],
        eval_dataset=enc["validation"],
        tokenizer=tok,
        compute_metrics=compute_metrics_mc,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
    )

    trainer.train()
    eval_res = trainer.evaluate()

    # save final
    save_dir = os.path.join(out_dir, "final_model")
    os.makedirs(save_dir, exist_ok=True)
    trainer.save_model(save_dir)
    tok.save_pretrained(save_dir)

    # report
    pred_out = trainer.predict(enc["validation"])
    preds = np.argmax(pred_out.predictions, axis=1)
    y_true = pred_out.label_ids
    rep = classification_report(y_true, preds, target_names=[id2label[i] for i in range(len(id2label))], digits=4)
    with open(os.path.join(out_dir, "classification_report.txt"), "w", encoding="utf-8") as f:
        f.write(rep)

    with open(os.path.join(out_dir, "label_map.json"), "w", encoding="utf-8") as f:
        json.dump({"id2label": id2label, "label2id": label2id}, f, ensure_ascii=False, indent=2)

    return save_dir, eval_res, id2label, {v:k for k,v in id2label.items()}

# ---- Train Scope
scope_dir, scope_eval, scope_i2l, scope_l2i = train_multiclass_from_jsonl(
    BASE_MODEL, paths["train_scope"], paths["valid_scope"],
    out_dir=os.path.join(OUT_ROOT, "scope"),
    epochs=EPOCHS, lr=LR, batch=BATCH, max_len=MAX_LEN
)
print("\n== Eval Scope ==", scope_eval)

# ---- Train Time
time_dir, time_eval, time_i2l, time_l2i = train_multiclass_from_jsonl(
    BASE_MODEL, paths["train_time"], paths["valid_time"],
    out_dir=os.path.join(OUT_ROOT, "time"),
    epochs=EPOCHS, lr=LR, batch=BATCH, max_len=MAX_LEN
)
print("== Eval Time  ==", time_eval)

# ---- Train Case
case_dir, case_eval, case_i2l, case_l2i = train_multiclass_from_jsonl(
    BASE_MODEL, paths["train_case"], paths["valid_case"],
    out_dir=os.path.join(OUT_ROOT, "case"),
    epochs=EPOCHS, lr=LR, batch=BATCH, max_len=MAX_LEN
)
print("== Eval Case  ==", case_eval)

# ====== (6) RUNNER (gabungan 3 head + route_hints) ======
from transformers import AutoConfig, AutoTokenizer, AutoModelForSequenceClassification

class DualTripleRunner:
    def __init__(self, scope_model_dir: str, time_model_dir: str, case_model_dir: str):
        # Scope
        self.scope_tok = AutoTokenizer.from_pretrained(scope_model_dir)
        self.scope_cfg = AutoConfig.from_pretrained(scope_model_dir)
        self.scope_i2l = self.scope_cfg.id2label
        self.scope_m = AutoModelForSequenceClassification.from_pretrained(scope_model_dir).eval()

        # Time
        self.time_tok = AutoTokenizer.from_pretrained(time_model_dir)
        self.time_cfg = AutoConfig.from_pretrained(time_model_dir)
        self.time_i2l = self.time_cfg.id2label
        self.time_m = AutoModelForSequenceClassification.from_pretrained(time_model_dir).eval()

        # Case
        self.case_tok = AutoTokenizer.from_pretrained(case_model_dir)
        self.case_cfg = AutoConfig.from_pretrained(case_model_dir)
        self.case_i2l = self.case_cfg.id2label
        self.case_m = AutoModelForSequenceClassification.from_pretrained(case_model_dir).eval()

        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.scope_m.to(self.device); self.time_m.to(self.device); self.case_m.to(self.device)

    @staticmethod
    def _decode(i2l, idx: int):
        if isinstance(i2l, dict):
            return i2l.get(idx) or i2l.get(str(idx)) or f"label_{idx}"
        try:
            return i2l[idx]
        except Exception:
            return f"label_{idx}"

    @torch.no_grad()
    def predict(self, text: str, max_len=128):
        # Scope
        s = self.scope_tok(text, truncation=True, padding=True, max_length=max_len, return_tensors="pt").to(self.device)
        s_logits = self.scope_m(**s).logits.squeeze(0).cpu().numpy()
        s_id = int(s_logits.argmax())
        s_label = self._decode(self.scope_i2l, s_id)
        s_conf = float(torch.softmax(torch.from_numpy(s_logits), dim=0)[s_id])

        # Time
        t = self.time_tok(text, truncation=True, padding=True, max_length=max_len, return_tensors="pt").to(self.device)
        t_logits = self.time_m(**t).logits.squeeze(0).cpu().numpy()
        t_id = int(t_logits.argmax())
        t_label = self._decode(self.time_i2l, t_id)
        t_conf = float(torch.softmax(torch.from_numpy(t_logits), dim=0)[t_id])

        # Case
        c = self.case_tok(text, truncation=True, padding=True, max_length=max_len, return_tensors="pt").to(self.device)
        c_logits = self.case_m(**c).logits.squeeze(0).cpu().numpy()
        c_id = int(c_logits.argmax())
        c_label = self._decode(self.case_i2l, c_id)
        c_conf = float(torch.softmax(torch.from_numpy(c_logits), dim=0)[c_id])

        route = []
        if c_label.startswith("RANGE"):
            route.append("use_range_extractor")
        if t_label in {"HOUR","DAY","WEEK","MONTH","YEAR"}:
            route.append(f"slot_{t_label.lower()}_extractor")
        if s_label == "ALL":
            route.append("aggregate_all_machines")
        else:
            route.append("machine_entity_extractor")

        return {
            "scope": {"label": s_label, "confidence": round(s_conf, 4)},
            "time":  {"label": t_label, "confidence": round(t_conf, 4)},
            "case":  {"label": c_label, "confidence": round(c_conf, 4)},
            "route_hints": route
        }

# ====== (7) DEMO INFERENCE ======
scope_dir = os.path.join(OUT_ROOT, "scope", "final_model")
time_dir  = os.path.join(OUT_ROOT, "time",  "final_model")
case_dir  = os.path.join(OUT_ROOT, "case",  "final_model")
runner = DualTripleRunner(scope_dir, time_dir, case_dir)

examples = [
    "Tolong tampilkan ringkasan semua mesin dari november sampai desember tahun lalu ini",
    "Cek tren mesin XP888A hari ini",
    "Apakah ada fluktuasi besar pada mesin XP888A dan PQ667 bulan ini?",
    "Mohon tampilkan nilai tertinggi semua mesin tahun lalu",
    "Tunjukkan tren mesin di bulan April tahun lalu dari awal bulan sampai pertengahan",
]
print("\n=== DEMO PREDIKSI ===")
for q in examples:
    try:
        print("\nQ:", q)
        print(runner.predict(q))
    except Exception as e:
        print("Lewati (pastikan model terlatih & path benar):", e)

print("\nSelesai. Model tersimpan di:", OUT_ROOT)


=== Cek file input ===
 - JSONL: True /workspace/train_trend_C_complex.jsonl
 - CSV  : True /workspace/labels_trend_C_index.enriched.csv

=== Preview data (3 baris JSONL & 6 baris CSV) ===
[JSONL] {"text": "Grafik mesin TRN-04 tanggal 5–17 Januari 2024.", "label": "c7", "slots": {"scope": "SINGLE", "machines": ["TRN-04"], "granularity": "DAY", "range_start": {"year": 2024, "month": 1, "day": 5}, "range_end": {"year": 2024, "month": 1, "day": 17}}}
[JSONL] {"text": "Bandingkan tren mesin XP888A dan XR101 dari November sampai Desember tahun lalu.", "label": "c4", "slots": {"scope": "MULTI", "machines": ["XP888A", "XR101"], "granularity": "MONTH", "range_start": {"year": "last_year", "month": 11}, "range_end": {"year": "last_year", "month": 12}}}
[JSONL] {"text": "Perlihatkan tren mesin XP888A dari Maret hingga Mei 2023.", "label": "c3", "slots": {"scope": "SINGLE", "machines": ["XP888A"], "granularity": "MONTH", "range_start": {"year": 2023, "month": 3}, "range_end": {"year": 2023, "mont

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/1478 [00:00<?, ? examples/s]

Map:   0%|          | 0/202 [00:00<?, ? examples/s]

...

Map:   0%|          | 0/1478 [00:00<?, ? examples/s]

Map:   0%|          | 0/202 [00:00<?, ? examples/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at indobenchmark/indobert-base-p1 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipykernel_5320/1868962970.py:210: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
Using EarlyStoppingCallback without load_best_model_at_end=True. Once training is finished, the best model will not be loaded automatically.


Epoch,Training Loss,Validation Loss,Acc,F1 Macro
1,0.171800,0.000907,1.000000,1.000000
2,0.000900,0.000514,1.000000,1.000000
3,0.000600,0.000370,1.000000,1.000000


== Eval Case  == {'eval_loss': 0.0003700499073602259, 'eval_acc': 1.0, 'eval_f1_macro': 1.0, 'eval_runtime': 0.4936, 'eval_samples_per_second': 409.256, 'eval_steps_per_second': 14.182, 'epoch': 3.0}

=== DEMO PREDIKSI ===

Q: Tolong tampilkan ringkasan semua mesin dari november sampai desember tahun lalu ini
{'scope': {'label': 'ALL', 'confidence': 0.9999}, 'time': {'label': 'ALL', 'confidence': 0.7819}, 'case': {'label': 'RANGE_REL', 'confidence': 0.8879}, 'route_hints': ['use_range_extractor', 'aggregate_all_machines']}

Q: Cek tren mesin XP888A hari ini
{'scope': {'label': 'NOT_ALL', 'confidence': 1.0}, 'time': {'label': 'MONTH', 'confidence': 0.9613}, 'case': {'label': 'RANGE_REL', 'confidence': 0.9923}, 'route_hints': ['use_range_extractor', 'slot_month_extractor', 'machine_entity_extractor']}

Q: Apakah ada fluktuasi besar pada mesin XP888A dan PQ667 bulan ini?
{'scope': {'label': 'NOT_ALL', 'confidence': 1.0}, 'time': {'label': 'ALL', 'confidence': 0.9992}, 'case': {'label': 'S

In [34]:
# =========================================================
# Indonesian Time & Entity Extractor (for Triple-Head output)
# - Converts phrases like "november sampai desember tahun lalu",
#   "senin", "bulan ini", "tahun lalu" into absolute date ranges
# - Extracts machine codes e.g. XP888A, PQ667
# - Merges with model predictions (scope/time/case/route_hints)
#   into a structured intent dict
# Assumptions:
#   - Timezone: Asia/Jakarta
#   - "today" is 2025-10-23 (can be changed via NOW)
# =========================================================
from __future__ import annotations
import re
from dataclasses import dataclass, asdict
from datetime import datetime, timedelta
import calendar

# ---- Config ----
NOW = datetime(2025, 10, 23)  # Asia/Jakarta today; ganti jika diperlukan

# ---- Helpers ----
ID_MONTHS = {
    "januari":1, "februari":2, "maret":3, "april":4, "mei":5, "juni":6,
    "juli":7, "agustus":8, "september":9, "oktober":10, "november":11, "desember":12
}
ID_DAYS = {
    "senin":0, "selasa":1, "rabu":2, "kamis":3, "jumat":4, "jum’at":4, "sabtu":5, "minggu":6, "ahad":6
}
def month_range(year:int, month:int):
    last_day = calendar.monthrange(year, month)[1]
    return datetime(year, month, 1), datetime(year, month, last_day, 23, 59, 59)

def year_range(year:int):
    return datetime(year, 1, 1), datetime(year, 12, 31, 23, 59, 59)

def clip_to_now(start:datetime, end:datetime, clip_to_today:bool=True):
    if clip_to_today and end > NOW:
        end = NOW
    return start, end

def prev_weekday(target_weekday:int, ref:datetime=NOW):
    # Return most recent <= ref day that matches target_weekday (Mon=0..Sun=6)
    delta = (ref.weekday() - target_weekday) % 7
    return (ref - timedelta(days=delta)).replace(hour=0, minute=0, second=0, microsecond=0), \
           (ref - timedelta(days=delta)).replace(hour=23, minute=59, second=59, microsecond=0)

# ---- Entity extractor (simple) ----
MACHINE_PAT = re.compile(r"\b[A-Z]{2,}[A-Z0-9]*\d+[A-Z0-9]*\b")

def extract_machines(text:str):
    # dedup while preserving order
    seen, out = set(), []
    for m in MACHINE_PAT.findall(text.upper()):
        if m not in seen:
            seen.add(m); out.append(m)
    return out

# ---- Time extractor ----
@dataclass
class TimeExtraction:
    kind: str              # "RANGE","PERIOD","POINT"
    start: datetime | None
    end: datetime | None
    granularity: str | None     # "HOUR/DAY/WEEK/MONTH/YEAR/ALL"
    note: str | None = None     # any heuristic note

# --- helper baru: end-of-day untuk NOW ---
def end_of_day(dt: datetime) -> datetime:
    return dt.replace(hour=23, minute=59, second=59, microsecond=0)

# --- perbaikan: urutan prioritas + EOD handling ---
def parse_time_id(text:str) -> TimeExtraction | None:
    t = text.lower().strip()

    # 1) RANGE BULAN (prioritas tertinggi)  e.g. "november sampai desember (tahun lalu/ini/2024)"
    m_range = re.search(
        r"\b(" + "|".join(ID_MONTHS.keys()) + r")\b.*?\b(s/d|sd|sampai|hingga|-)\b.*?\b(" +
        "|".join(ID_MONTHS.keys()) + r")\b(?:\s+(tahun\s+lalu|tahun\s+ini|\d{4}))?",
        t
    )
    if m_range:
        m1 = ID_MONTHS[m_range.group(1)]
        m2 = ID_MONTHS[m_range.group(3)]
        tail = (m_range.group(4) or "").strip()
        if tail == "tahun lalu":
            y = NOW.year - 1
        elif tail == "tahun ini":
            y = NOW.year
        elif tail.isdigit() and len(tail)==4:
            y = int(tail)
        else:
            y = NOW.year - 1 if "tahun lalu" in t else NOW.year
        s1,e1 = month_range(y, m1)
        s2,e2 = month_range(y, m2)
        s, e = (s1 if s1 < s2 else s2), (e1 if e1 > e2 else e2)
        if y == NOW.year:
            # kalau tahun ini, batasi sampai EOD hari ini
            e = min(e, end_of_day(NOW))
        return TimeExtraction("RANGE", s, e, "MONTH", "range bulan")

    # 2) BULAN TUNGGAL (april [tahun lalu/ini/2024])
    m_single = re.search(
        r"\b(" + "|".join(ID_MONTHS.keys()) + r")\b(?:\s+(tahun\s+lalu|tahun\s+ini|\d{4}))?", t
    )
    if m_single:
        m = ID_MONTHS[m_single.group(1)]
        tail = (m_single.group(2) or "").strip()
        if tail == "tahun lalu":
            y = NOW.year - 1
        elif tail == "tahun ini":
            y = NOW.year
        elif tail.isdigit() and len(tail)==4:
            y = int(tail)
        else:
            y = NOW.year - 1 if "tahun lalu" in t else NOW.year
        s,e = month_range(y, m)
        if y == NOW.year:
            e = min(e, end_of_day(NOW))  # MTD
        return TimeExtraction("PERIOD", s, e, "MONTH", "bulan tunggal")

    # 3) BULAN INI / BULAN LALU (PERIOD)
    if "bulan ini" in t:
        s,e = month_range(NOW.year, NOW.month)
        e = min(e, end_of_day(NOW))     # MTD sampai akhir hari ini
        return TimeExtraction("PERIOD", s, e, "MONTH", "bulan ini (MTD)")

    if "bulan lalu" in t:
        y = NOW.year; m = NOW.month - 1
        if m == 0: y -= 1; m = 12
        s,e = month_range(y, m)
        return TimeExtraction("PERIOD", s, e, "MONTH", "bulan lalu")

    # 4) HARI DALAM MINGGU ("senin", "rabu", ...)
    for name, wd in ID_DAYS.items():
        if re.search(rf"\b{name}\b", t):
            s,e = prev_weekday(wd, NOW)
            return TimeExtraction("POINT", s, e, "DAY", f"hari {name}")

    # 5) HARI INI / KEMARIN
    if "hari ini" in t:
        s = NOW.replace(hour=0, minute=0, second=0, microsecond=0)
        e = NOW  # waktu sekarang; bisa juga end_of_day(NOW) jika maunya EOD
        return TimeExtraction("POINT", s, e, "DAY", "hari ini")

    if "kemarin" in t:
        d = NOW - timedelta(days=1)
        s = d.replace(hour=0, minute=0, second=0, microsecond=0)
        e = d.replace(hour=23, minute=59, second=59, microsecond=0)
        return TimeExtraction("POINT", s, e, "DAY", "kemarin")

    # 6) TAHUN LALU / TAHUN INI / TAHUN YYYY  (fallback tahunan)
    if "tahun lalu" in t:
        y = NOW.year - 1
        s,e = year_range(y)
        return TimeExtraction("RANGE", s, e, "YEAR", "tahun lalu")

    if "tahun ini" in t:
        y = NOW.year
        s,e = year_range(y)
        e = min(e, end_of_day(NOW))  # YTD
        return TimeExtraction("PERIOD", s, e, "YEAR", "tahun ini (YTD)")

    y_exp = re.search(r"tahun\s+(\d{4})", t)
    if y_exp:
        y = int(y_exp.group(1))
        s,e = year_range(y)
        if y == NOW.year:
            e = min(e, end_of_day(NOW))
        return TimeExtraction("PERIOD", s, e, "YEAR", "tahun eksplisit")

    # Tidak ditemukan
    return None

# --- interpret(): hanya ubah bagian route range flag ---
def interpret(query_text:str, model_pred:dict):
    time_info = parse_time_id(query_text)  # may be None
    machines = extract_machines(query_text)

    # final granularity from extractor (if present)
    if time_info:
        final_gran = time_info.granularity
        start = time_info.start.isoformat() if time_info.start else None
        end   = time_info.end.isoformat()   if time_info.end   else None
        note  = time_info.note
    else:
        final_gran = model_pred["time"]["label"]
        start = end = note = None

    scope_label = model_pred["scope"]["label"]
    scope = "ALL" if scope_label == "ALL" else "SUBSET"

    # route_hints (extractor-first)
    route = []
    if time_info and time_info.kind == "RANGE":
        route.append("use_range_extractor")
    if final_gran in {"HOUR","DAY","WEEK","MONTH","YEAR"}:
        route.append(f"slot_{final_gran.lower()}_extractor")
    route.append("aggregate_all_machines" if scope == "ALL" else "machine_entity_extractor")

    # ----- CASE FIX -----
    qlow = query_text.lower()
    case_label = model_pred["case"]["label"]
    # titik waktu tunggal -> SINGLE_POINT
    if time_info and time_info.kind == "POINT":
        case_label = "SINGLE_POINT"
    # periode tunggal (bulan ini/lalu, bulan eksplisit, tahun ini) -> SINGLE_PERIOD
    if time_info and time_info.kind == "PERIOD":
        # kecuali kalau ada kata "ringkasan/ringkas" eksplisit atau model mendeteksi COMPARE
        if "ringkasan" not in qlow and "ringkas" not in qlow and case_label != "COMPARE":
            case_label = "SINGLE_PERIOD"

    intent = {
        "query": query_text,
        "scope": scope,
        "machines": machines if scope == "SUBSET" else [],
        "granularity": final_gran,
        "time_range": {"start": start, "end": end, "note": note},
        "case": case_label,
        "route_hints": route,
        "meta": {
            "model_time_label": model_pred["time"]["label"],
            "model_time_conf":  model_pred["time"]["confidence"],
            "model_case_conf":  model_pred["case"]["confidence"],
            "model_scope_conf": model_pred["scope"]["confidence"],
            "extractor_kind": time_info.kind if time_info else None
        }
    }
    return intent




# ---- Demo on your four examples ----
examples = [
    ("Tolong tampilkan ringkasan semua mesin dari november sampai desember tahun lalu",
     {'scope': {'label': 'ALL', 'confidence': 0.9999}, 'time': {'label': 'MONTH', 'confidence': 0.5835},
      'case': {'label': 'RANGE_REL', 'confidence': 0.7075}, 'route_hints': ['use_range_extractor', 'slot_month_extractor', 'aggregate_all_machines']}),
    ("Cek tren mesin XP888A senin",
     {'scope': {'label': 'NOT_ALL', 'confidence': 1.0}, 'time': {'label': 'ALL', 'confidence': 0.7846},
      'case': {'label': 'RANGE_REL', 'confidence': 0.9888}, 'route_hints': ['use_range_extractor', 'machine_entity_extractor']}),
    ("Apakah ada fluktuasi besar pada mesin XP888A dan PQ667 bulan ini?",
     {'scope': {'label': 'NOT_ALL', 'confidence': 1.0}, 'time': {'label': 'ALL', 'confidence': 0.9992},
      'case': {'label': 'SUMMARY', 'confidence': 0.7034}, 'route_hints': ['machine_entity_extractor']}),
    ("Mohon tampilkan nilai tertinggi semua mesin tahun lalu",
     {'scope': {'label': 'ALL', 'confidence': 0.9999}, 'time': {'label': 'MONTH', 'confidence': 0.6923},
      'case': {'label': 'RANGE_REL', 'confidence': 0.7708}, 'route_hints': ['use_range_extractor', 'slot_month_extractor', 'aggregate_all_machines']}),
]

for q, pred in examples:
    out = interpret(q, pred)
    print("\nQ:", q)
    print(out)



Q: Tolong tampilkan ringkasan semua mesin dari november sampai desember tahun lalu
{'query': 'Tolong tampilkan ringkasan semua mesin dari november sampai desember tahun lalu', 'scope': 'ALL', 'machines': [], 'granularity': 'MONTH', 'time_range': {'start': '2024-11-01T00:00:00', 'end': '2024-12-31T23:59:59', 'note': 'range bulan'}, 'case': 'RANGE_REL', 'route_hints': ['use_range_extractor', 'slot_month_extractor', 'aggregate_all_machines'], 'meta': {'model_time_label': 'MONTH', 'model_time_conf': 0.5835, 'model_case_conf': 0.7075, 'model_scope_conf': 0.9999, 'extractor_kind': 'RANGE'}}

Q: Cek tren mesin XP888A senin
{'query': 'Cek tren mesin XP888A senin', 'scope': 'SUBSET', 'machines': ['XP888A'], 'granularity': 'DAY', 'time_range': {'start': '2025-10-20T00:00:00', 'end': '2025-10-20T23:59:59', 'note': 'hari senin'}, 'case': 'SINGLE_POINT', 'route_hints': ['slot_day_extractor', 'machine_entity_extractor'], 'meta': {'model_time_label': 'ALL', 'model_time_conf': 0.7846, 'model_case_con

In [52]:

examples = [
    ("Tolong tampilkan ringkasan semua mesin dari november sampai desember tahun lalu",
     {'scope': {'label': 'ALL', 'confidence': 0.9999}, 'time': {'label': 'MONTH', 'confidence': 0.5835},
      'case': {'label': 'RANGE_REL', 'confidence': 0.7075}, 'route_hints': ['use_range_extractor', 'slot_month_extractor', 'aggregate_all_machines']}),
    ("Cek tren mesin XP888A senin",
     {'scope': {'label': 'NOT_ALL', 'confidence': 1.0}, 'time': {'label': 'ALL', 'confidence': 0.7846},
      'case': {'label': 'RANGE_REL', 'confidence': 0.9888}, 'route_hints': ['use_range_extractor', 'machine_entity_extractor']}),
    ("Apakah ada fluktuasi besar pada mesin XP888A dan PQ667 januari tahun ini?",
     {'scope': {'label': 'NOT_ALL', 'confidence': 1.0}, 'time': {'label': 'ALL', 'confidence': 0.9992},
      'case': {'label': 'SUMMARY', 'confidence': 0.7034}, 'route_hints': ['machine_entity_extractor']}),
    ("Mohon tampilkan nilai tertinggi semua mesin tahun lalu",
     {'scope': {'label': 'ALL', 'confidence': 0.9999}, 'time': {'label': 'MONTH', 'confidence': 0.6923},
      'case': {'label': 'RANGE_REL', 'confidence': 0.7708}, 'route_hints': ['use_range_extractor', 'slot_month_extractor', 'aggregate_all_machines']}),

]
for q, pred in examples:
    out = interpret(q, pred)
    print("\nQ:", q)
    print(out)


Q: Tolong tampilkan ringkasan semua mesin dari november sampai desember tahun lalu
{'query': 'Tolong tampilkan ringkasan semua mesin dari november sampai desember tahun lalu', 'scope': 'ALL', 'machines': [], 'granularity': 'MONTH', 'time_range': {'start': '2024-11-01T00:00:00', 'end': '2024-12-31T23:59:59', 'note': 'range bulan'}, 'case': 'RANGE_REL', 'route_hints': ['use_range_extractor', 'slot_month_extractor', 'aggregate_all_machines'], 'meta': {'model_time_label': 'MONTH', 'model_time_conf': 0.5835, 'model_case_conf': 0.7075, 'model_scope_conf': 0.9999, 'extractor_kind': 'RANGE'}}

Q: Cek tren mesin XP888A senin
{'query': 'Cek tren mesin XP888A senin', 'scope': 'SUBSET', 'machines': ['XP888A'], 'granularity': 'DAY', 'time_range': {'start': '2025-10-20T00:00:00', 'end': '2025-10-20T23:59:59', 'note': 'hari senin'}, 'case': 'SINGLE_POINT', 'route_hints': ['slot_day_extractor', 'machine_entity_extractor'], 'meta': {'model_time_label': 'ALL', 'model_time_conf': 0.7846, 'model_case_con

In [66]:
import pandas as pd
import numpy as np
from dataclasses import dataclass
from typing import Optional, Dict, Any, List, Tuple
from datetime import datetime

# ===== helpers =====
def _freq_from_gran(gran: str) -> str:
    gran = (gran or "DAY").upper()
    return {
        "HOUR": "1H",
        "DAY": "1D",
        "WEEK": "1W-MON",   # minggu mulai Senin
        "MONTH": "1MS",     # Month Start
        "YEAR": "1Y",
        "ALL": "1D",        # fallback
    }.get(gran, "1D")

def _agg_op(scope: str) -> str:
    # Pilih agregasi sesuai metrik-mu:
    #  - "sum" untuk output/throughput
    #  - "mean" untuk temperatur, kualitas, dll.
    return "mean" if scope == "SUBSET" else "mean"

def _linreg_slope(x: np.ndarray, y: np.ndarray) -> float:
    # slope sederhana: cov(x,y)/var(x)
    x = x.astype(float)
    y = y.astype(float)
    if len(x) < 2 or np.allclose(x.var(), 0):
        return 0.0
    return np.cov(x, y, bias=True)[0,1] / np.var(x)

def _pct_change(a: float, b: float) -> Optional[float]:
    # % change dari a -> b (mis. rata-rata window lama -> baru)
    if a is None or b is None or a == 0:
        return None
    return (b - a) / abs(a)

def _max_drawdown(series: pd.Series) -> float:
    # maksimum penurunan relatif dari puncak ke lembah berikutnya
    cummax = series.cummax()
    dd = (series - cummax) / cummax.replace(0, np.nan)
    return float(dd.min()) if len(series) else 0.0

def _zscore_outliers(series: pd.Series, z=3.0) -> List[pd.Timestamp]:
    if len(series) < 5 or series.std(ddof=0) == 0:
        return []
    zs = (series - series.mean())/series.std(ddof=0)
    return list(series.index[(zs.abs() >= z)])
    
def extract_subintent(query: str) -> str:
    q = query.lower()
    has_trend = any(k in q for k in ["tren", "trend", "fluktuasi"])
    has_min = "min" in q or "minimum" in q or "terendah" in q
    has_max = "max" in q or "maksimum" in q or "tertinggi" in q
    has_summary = "ringkasan" in q or "summary" in q

    if has_summary or (has_min and has_max and has_trend):
        return "summary"
    if has_trend and has_min and not has_max:
        return "trend_min"
    if has_trend and has_max and not has_min:
        return "trend_max"
    if has_min and has_max and not has_trend:
        return "minmax"
    if has_trend and not (has_min or has_max):
        return "trend"
    if has_min and not has_trend:
        return "min_only"
    if has_max and not has_trend:
        return "max_only"
    return "summary"

# ===== core analyzer =====
import pandas as pd
import numpy as np

def _auto_freq(start, end):
    """Heuristik frekuensi dari lebar rentang."""
    if start is None or end is None:
        return "1D"
    span = (pd.Timestamp(end) - pd.Timestamp(start)).total_seconds()
    # <= 3 hari → 1H ; <= 35 hari → 1D ; <= 200 hari → 1W ; lainnya → 1MS
    if span <= 3*24*3600:      return "1H"
    if span <= 35*24*3600:     return "1D"
    if span <= 200*24*3600:    return "1W-MON"
    return "1MS"

def analyze_trend(
    df: pd.DataFrame,
    intent: dict,
    ts_col="ts",
    machine_col="machine_id",
    value_col="value",
    agg=None,
    slope_threshold=0.0001,
    recent_ratio=0.3,
    subintent: str = "summary",
):
    assert all(c in df.columns for c in [ts_col, machine_col, value_col]), "Kolom ts/machine_id/value wajib ada"

    start = intent["time_range"]["start"]
    end   = intent["time_range"]["end"]
    scope = intent["scope"]
    gran  = (intent.get("granularity") or "DAY").upper()
    machines = intent.get("machines", []) or []

    df = df.copy()
    df[ts_col] = pd.to_datetime(df[ts_col], errors="coerce")
    df = df.dropna(subset=[ts_col, value_col])

    if start: df = df[df[ts_col] >= pd.Timestamp(start)]
    if end:   df = df[df[ts_col] <= pd.Timestamp(end)]

    if scope == "SUBSET" and machines:
        df = df[df[machine_col].isin(machines)]

    if df.empty:
        return {"empty": True, "reason": "No data in range/selection", "intent": intent}

    # Pilih freq: gunakan granularity jika ‘masuk akal’, jika tidak auto turun ke yang lebih halus
    pref = {"HOUR":"1H","DAY":"1D","WEEK":"1W-MON","MONTH":"1MS","YEAR":"1Y","ALL":"1D"}.get(gran, "1D")
    auto = _auto_freq(start, end)
    # Ambil yang lebih halus di antara keduanya
    order = ["1H","1D","1W-MON","1MS","1Y"]
    freq = pref if order.index(pref) < order.index(auto) else auto

    # Agregasi antar mesin
    if agg is None:
        agg = "mean"   # ganti "sum" kalau metrik adalah throughput

    g = (df[[ts_col, value_col]]
         .set_index(ts_col)
         .resample(freq)
         .agg({value_col: agg})
         .dropna())

    # Jika user cuma minta min/max, kita tidak butuh >= 2 titik
    sub = (subintent or "").lower()
    needs_trend = any(k in sub for k in ["trend", "summary"]) or sub == ""  # summary mengandung trend
    needs_min = ("min" in sub) or (sub in {"summary","min_only","minmax"})
    needs_max = ("max" in sub) or (sub in {"summary","max_only","minmax"})

    if len(g) < 2 and needs_trend:
        # turunkan frekuensi lagi kalau bisa
        if freq == "1MS":
            g = g.resample("1D").mean().dropna()
        elif freq == "1W-MON":
            g = g.resample("1D").mean().dropna()
        elif freq == "1D":
            g = g.resample("1H").mean().dropna()

    if len(g) < 2 and needs_trend:
        # Masih kurang titik, tapi tetap bisa jawab min/max jika diminta
        if needs_min or needs_max:
            min_val = float(g[value_col].min()) if len(g) else float("nan")
            max_val = float(g[value_col].max()) if len(g) else float("nan")
            tmin = g[value_col].idxmin().isoformat() if len(g) else None
            tmax = g[value_col].idxmax().isoformat() if len(g) else None
            return {
                "intent": intent,
                "granularity_used": freq,
                "series_points": int(len(g)),
                "summary_text": (
                    (f"Min {min_val:.3f} ({tmin[:10]}). " if needs_min and tmin else "") +
                    (f"Max {max_val:.3f} ({tmax[:10]})."  if needs_max and tmax else "")
                ).strip() or "Tidak cukup titik untuk tren.",
                "summary": {
                    "direction": None, "pct_change": None,
                    "min": {"value": min_val, "ts": tmin},
                    "max": {"value": max_val, "ts": tmax},
                },
                "data_preview": g.tail(5).reset_index().to_dict(orient="records")
            }
        return {"empty": True, "reason": "Too few points after resampling", "intent": intent}

    # --- hitung metrik tren
    x = np.arange(len(g))
    y = g[value_col].values
    slope = float(np.cov(x, y, bias=True)[0,1] / (np.var(x) if np.var(x) != 0 else 1))

    split = max(1, int(len(g)*(1-0.3)))
    prev = g[value_col].iloc[:split]; recent = g[value_col].iloc[split:]
    mean_prev = float(prev.mean()) if len(prev) else None
    mean_recent = float(recent.mean()) if len(recent) else None
    pct = ( (mean_recent - mean_prev)/abs(mean_prev) ) if (mean_prev not in (None,0)) else None

    direction = "flat"
    if pct is not None:
        if pct > 0.02 or slope > slope_threshold: direction = "up"
        if pct < -0.02 or slope < -slope_threshold: direction = "down"
        if abs(pct) <= 0.01 and abs(slope) <= slope_threshold: direction = "flat"

    min_val = float(g[value_col].min()); max_val = float(g[value_col].max())
    tmin = g[value_col].idxmin().isoformat(); tmax = g[value_col].idxmax().isoformat()

    # compose teks sesuai subintent
    parts = []
    if needs_trend:
        parts.append(f"Tren { {'up':'meningkat','down':'menurun','flat':'stabil'}[direction] }"
                     + (f" ({pct*100:.1f}% change)" if pct is not None else "" ) + ".")
    if needs_min:
        parts.append(f"Min {min_val:.3f} ({tmin[:10]}).")
    if needs_max:
        parts.append(f"Max {max_val:.3f} ({tmax[:10]}).")
    # --- tambahkan ini setelah menghitung min/max & direction ---

    # # hitung fluktuasi (volatilitas & outlier)
    # std = float(g[value_col].std(ddof=0))
    # mean = float(g[value_col].mean())
    # cv = float(std / mean) if mean != 0 else 0.0  # coefficient of variation
    
    # # deteksi spike/outlier sederhana
    # zscore = (g[value_col] - g[value_col].mean()) / (g[value_col].std(ddof=0) or 1)
    # spikes = g[value_col][abs(zscore) >= 3]  # nilai ekstrem ±3σ
    # n_spikes = len(spikes)
    
    # # --- tentukan apakah fluktuasi besar ---
    # fluktuasi_besar = (cv > 0.1) or (n_spikes >= 1)  # threshold bisa diatur

        # --- Volatilitas & Outlier detail ---
    std = float(g[value_col].std(ddof=0))
    mean = float(g[value_col].mean())
    cv = float(std / mean) if mean != 0 else 0.0

    # Z-score untuk tiap titik
    den = g[value_col].std(ddof=0)
    den = den if den and den != 0 else 1.0
    zs = (g[value_col] - g[value_col].mean()) / den
    
    # Outlier = |z| >= 3
    mask_spike = zs.abs() >= 3
    spike_series = g.loc[mask_spike, value_col]
    n_spikes = int(mask_spike.sum())

    # Simpan detail (urutkan menurut |z| desc, batasi 10 titik)
    outlier_points = []
    if n_spikes > 0:
        tmp = zs[mask_spike].abs().sort_values(ascending=False)
        for ts in tmp.index[:10]:
            outlier_points.append({
                "ts": ts.isoformat(),
                "value": float(g.loc[ts, value_col]),
                "z": float(zs.loc[ts])
            })
    
    # Flag fluktuasi besar (atur threshold sesuai kebutuhan)
    fluktuasi_besar = (cv > 0.10) or (n_spikes >= 1)

    # ---------- Compose text sesuai subintent ----------
    parts = []
    if needs_trend:
        dir_map = {"up":"meningkat","down":"menurun","flat":"stabil", None:"-"}
        if fluktuasi_besar:
            parts.append(
                f"Terdapat fluktuasi besar (CV={cv:.6f}, outlier={n_spikes})."
            )
        else:
            pct_txt = f"{(pct*100):.10f}%" if pct is not None else "NA"
            parts.append(
                f"Tren {dir_map.get(direction)} ({pct_txt}). (CV={cv:.6f})."
            )
    if needs_min:
        parts.append(f"Min {min_val:.10f} ({tmin[:10]}).")
    if needs_max:
        parts.append(f"Max {max_val:.10f} ({tmax[:10]}).")
    
    return {
        "intent": intent,
        "series_points": int(len(g)),
        "granularity_used": freq,
        "summary_text": " ".join(parts).strip(),
        "summary": {
            "direction": direction,
            "pct_change": pct,
            "min": {"value": min_val, "ts": tmin},
            "max": {"value": max_val, "ts": tmax},
            "volatility": {"std": std, "mean": mean, "cv": cv},
            "outliers": {
                "count": n_spikes,
                "points": outlier_points  # <-- detail outlier
            },
        },
        "data_preview": g.tail(5).reset_index().to_dict(orient="records"),
    }





    

    # if needs_trend:
    #     if fluktuasi_besar:
    #         parts.append(f"Terdapat fluktuasi besar (CV={cv:.3f}, outlier={n_spikes}).")
    #     else:
    #         parts.append(
    #             f"Tren { {'up':'meningkat','down':'menurun','flat':'stabil'}[direction] }"
    #             + (f" ({pct*100:.10f}% change)" if pct is not None else "")
    #             + f" (CV={cv:.3f})."
    #         )
    # return {
    #     "intent": intent,
    #     "series_points": int(len(g)),
    #     "granularity_used": freq,
    #     "summary_text": " ".join(parts) or "OK",
    #     "summary" : {
    #         "direction": direction,
    #         "pct_change": pct,
    #         "min": {"value": min_val, "ts": tmin},
    #         "max": {"value": max_val, "ts": tmax},
    #         "volatility": {"std": std, "cv": cv},
    #         "outliers": n_spikes,
    #     },
    #     "data_preview": g.tail(5).reset_index().to_dict(orient="records"),
    # }


def format_trend_response(result, subintent: str):
    if result.get("empty"):
        return f"Tidak ada data ({result['reason']})"

    s = result["summary"]
    text_parts = []

    if "trend" in subintent or subintent == "summary":
        text_parts.append(f"Tren {s['direction']} ({s['pct_change']*100:.1f}% perubahan).")

    if "min" in subintent or subintent in {"summary", "min_only", "minmax"}:
        text_parts.append(f"Nilai minimum {s['min']['value']:.2f} pada {s['min']['ts'][:10]}.")

    if "max" in subintent or subintent in {"summary", "max_only", "minmax"}:
        text_parts.append(f"Nilai maksimum {s['max']['value']:.2f} pada {s['max']['ts'][:10]}.")

    if subintent == "summary":
        text_parts.append(f"Volatilitas {s['volatility']['cv']:.2f}, drawdown {s['max_drawdown']:.2%}.")

    return " ".join(text_parts)


def pretty_response(result: dict, subintent: str, places=10, decimal_comma=True, show_outliers=3):
    import numpy as np
    def fmt_num(x, places=10, percent=False, decimal_comma=True):
        if x is None or (isinstance(x, float) and (np.isnan(x) or np.isinf(x))):
            return "NA"
        if percent: x = x * 100.0
        s = f"{x:.{places}f}"
        return s.replace(".", ",") if decimal_comma else s

    if result.get("empty"):
        return f"Tidak ada data ({result.get('reason','')})"

    s = result["summary"]
    sub = (subintent or "summary").lower()
    needs_trend = ("trend" in sub) or (sub == "summary")
    needs_min   = ("min" in sub) or (sub in {"summary","min_only","minmax"})
    needs_max   = ("max" in sub) or (sub in {"summary","max_only","minmax"})

    parts = []
    if needs_trend:
        dir_map = {"up":"meningkat","down":"menurun","flat":"stabil", None:"-"}
        pct = s.get("pct_change")
        cv  = s["volatility"]["cv"]
        flg = (cv > 0.10) or (s.get("outliers",{}).get("count",0) >= 1)
        if flg:
            parts.append(f"Terdapat fluktuasi besar (CV={fmt_num(cv,6,False,decimal_comma)}, outlier={s['outliers']['count']}).")
        else:
            parts.append(
                f"Tren {dir_map.get(s.get('direction'))} "
                f"({fmt_num(pct, places, True, decimal_comma)}%). "
                f"CV={fmt_num(cv,6,False,decimal_comma)}."
            )

    if needs_min and s.get("min"):
        parts.append(f"Min {fmt_num(s['min']['value'], places, False, decimal_comma)} ({str(s['min']['ts'])[:10]}).")
    if needs_max and s.get("max"):
        parts.append(f"Max {fmt_num(s['max']['value'], places, False, decimal_comma)} ({str(s['max']['ts'])[:10]}).")

    # tampilkan contoh outlier detail
    pts = s.get("outliers",{}).get("points",[]) or []
    if pts:
        lines = []
        for p in pts[:show_outliers]:
            lines.append(
                f"- {p['ts'][:19]} | nilai={fmt_num(p['value'], places, False, decimal_comma)} | z={fmt_num(p['z'], 4, False, decimal_comma)}"
            )
        parts.append("Contoh outlier:\n" + "\n".join(lines))

    return " ".join(parts)




In [39]:
import pandas as pd
import numpy as np
from pathlib import Path

# ====== CONFIG ======
EXCEL_PATH = "/workspace/Machine_Value_Processed.xlsx"   # ganti ke path file kamu
SHEET_NAME = "SW_3min"                    # sheet yang kamu sebut
SAVE_CSV   = "/workspace/df_raw.csv"     # opsional: simpan ke CSV

# ====== LOAD ======
# read_excel otomatis pilih engine (openpyxl/xlrd) sesuai tipe file
raw = pd.read_excel(EXCEL_PATH, sheet_name=SHEET_NAME)

# ====== NORMALISASI NAMA KOLOM ======
def norm(s: str) -> str:
    return (s or "").strip().lower().replace(" ", "").replace("_", "")

cols_norm = {c: norm(str(c)) for c in raw.columns}

# Kandidat nama kolom (beberapa kemungkinan dari screenshot)
# - machine/tag: "tagnumber", "tagnumbe" (typo), "tag", "machine", "mesin"
# - timestamp  : "timestamp/datetime", "timestamp", "datetime", "waktu"
# - value      : "value", "nilai"
col_map = {}

# cari kolom machine
for c, n in cols_norm.items():
    if n in {"tagnumber","tagnumbe","tag","machine","mesin","unit"}:
        col_map["machine_id"] = c
        break

# cari kolom timestamp
for c, n in cols_norm.items():
    if n in {"timestamp/datetime","timestampdatetime","timestamp","datetime","waktu","timestamptime"}:
        col_map["ts"] = c
        break

# cari kolom value
for c, n in cols_norm.items():
    if n in {"value","nilai"}:
        col_map["value"] = c
        break

# validasi
missing = {"machine_id","ts","value"} - set(col_map.keys())
if missing:
    raise ValueError(f"Kolom wajib tidak ditemukan: {missing}. Kolom tersedia: {list(raw.columns)}")

# ====== BENTUK DF_RAW ======
df_raw = raw[[col_map["machine_id"], col_map["ts"], col_map["value"]]].copy()
df_raw.columns = ["machine_id", "ts", "value"]

# 1) parse waktu
df_raw["ts"] = pd.to_datetime(df_raw["ts"], errors="coerce")

# 2) angka dengan koma sebagai desimal → ganti koma jadi titik dulu
#    (kalau sudah titik, ini tidak mengubah apa-apa)
def to_float_any(x):
    if isinstance(x, str):
        x = x.replace(",", ".")
    return pd.to_numeric(x, errors="coerce")

df_raw["value"] = df_raw["value"].apply(to_float_any)

# 3) bersihkan
df_raw = df_raw.dropna(subset=["ts","value"]).astype({"machine_id": "string"})
df_raw = df_raw.sort_values(["machine_id", "ts"]).drop_duplicates(subset=["machine_id","ts"], keep="last").reset_index(drop=True)

print("df_raw (head):")
print(df_raw.head())

# ====== (OPSIONAL) FITUR TAMBAHAN ala screenshot (lag 1..6) ======
# ini tidak wajib, tapi kalau kamu perlu kolom y_lag_1..y_lag_6 per mesin:
def add_lags(group, max_lag=6):
    g = group.sort_values("ts").copy()
    for k in range(1, max_lag+1):
        g[f"y_lag_{k}"] = g["value"].shift(k)
    return g

df_feat = df_raw.groupby("machine_id", group_keys=False).apply(add_lags, max_lag=6)

print("\nDengan fitur lag (head):")
print(df_feat.head(10))

# ====== (OPSIONAL) SIMPAN ======
Path(SAVE_CSV).parent.mkdir(parents=True, exist_ok=True)
df_raw.to_csv(SAVE_CSV, index=False)
print(f"\nSaved df_raw → {SAVE_CSV}")


df_raw (head):
  machine_id                  ts       value
0      PQ667 2025-01-01 12:00:57  127.723783
1      PQ667 2025-01-01 12:03:57  127.749400
2      PQ667 2025-01-01 12:06:57  127.749400
3      PQ667 2025-01-01 12:09:57  133.227233
4      PQ667 2025-01-01 12:12:57  138.705067

Dengan fitur lag (head):
  machine_id                  ts       value     y_lag_1     y_lag_2  \
0      PQ667 2025-01-01 12:00:57  127.723783         NaN         NaN   
1      PQ667 2025-01-01 12:03:57  127.749400  127.723783         NaN   
2      PQ667 2025-01-01 12:06:57  127.749400  127.749400  127.723783   
3      PQ667 2025-01-01 12:09:57  133.227233  127.749400  127.749400   
4      PQ667 2025-01-01 12:12:57  138.705067  133.227233  127.749400   
5      PQ667 2025-01-01 12:15:57  144.182900  138.705067  133.227233   
6      PQ667 2025-01-01 12:18:57  144.182900  144.182900  138.705067   
7      PQ667 2025-01-01 12:21:57  135.917700  144.182900  144.182900   
8      PQ667 2025-01-01 12:24:57  127.652

/tmp/ipykernel_5320/2894117815.py:80: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_feat = df_raw.groupby("machine_id", group_keys=False).apply(add_lags, max_lag=6)


In [76]:
import pandas as pd
import numpy as np

# =========================
#  Utilities
# =========================
def _auto_freq(start, end):
    if start is None or end is None:
        return "1D"
    span = (pd.Timestamp(end) - pd.Timestamp(start)).total_seconds()
    if span <= 3*24*3600:      return "1H"     # ≤ 3 hari
    if span <= 35*24*3600:     return "1D"     # ≤ 5 minggu
    if span <= 200*24*3600:    return "1W-MON" # ≤ ~6.5 bulan
    return "1MS"                               

def _trend_stats(series: pd.Series):
    if len(series) < 2:
        return {"slope": 0.0, "pct": None, "direction": None}
    y = series.values.astype(float)
    x = np.arange(len(series), dtype=float)
    varx = np.var(x)
    slope = float(np.cov(x, y, bias=True)[0,1] / (varx if varx != 0 else 1))
    split = max(1, int(len(series)*0.7))
    prev = series.iloc[:split]
    recent = series.iloc[split:]
    pct = ((recent.mean()-prev.mean())/abs(prev.mean())) if prev.mean() not in (0, np.nan, None, 0.0) else None
    direction = "flat"
    if pct is not None:
        if pct > 0.02 or slope > 1e-4: direction = "up"
        if pct < -0.02 or slope < -1e-4: direction = "down"
        if abs(pct) <= 0.01 and abs(slope) <= 1e-4: direction = "flat"
    return {"slope": float(slope), "pct": None if pct is None else float(pct), "direction": direction}

def _mad(x: pd.Series):
    med = x.median()
    mad = (x - med).abs().median()
    return med, (mad if mad and mad != 0 else 1.0)

def _rolling_z(series: pd.Series, win: int = 12):
    """rolling z-score (mean/std di-window), tahan seasonal kecil"""
    if len(series) < max(5, win):
        return pd.Series(index=series.index, dtype=float)
    m = series.rolling(win, min_periods=max(3, win//3)).mean()
    s = series.rolling(win, min_periods=max(3, win//3)).std(ddof=0).replace(0, np.nan)
    return (series - m) / s

def _top_fluctuation_windows(series: pd.Series, win: int, k: int = 3):
    """cari periode dengan volatilitas (std/CV) tertinggi"""
    if len(series) < max(5, win):
        return []
    roll_std = series.rolling(win, min_periods=max(3, win//3)).std(ddof=0)
    roll_mean = series.rolling(win, min_periods=max(3, win//3)).mean().replace(0, np.nan)
    roll_cv = (roll_std / roll_mean).replace([np.inf, -np.inf], np.nan)
    # ambil top-k window center
    sc = roll_cv.fillna(0)
    idxs = sc.nlargest(k).index
    out = []
    half = max(1, win//2)
    for ts in idxs:
        try:
            s = series.loc[:ts].iloc[-win:]
        except Exception:
            continue
        if s.empty: 
            continue
        out.append({
            "start": s.index[0].isoformat(),
            "end": s.index[-1].isoformat(),
            "std": float(s.std(ddof=0)),
            "mean": float(s.mean()),
            "cv": float((s.std(ddof=0) / (s.mean() if s.mean()!=0 else np.nan)) if s.mean()!=0 else np.nan),
        })
    return out

# =========================
#  Per-machine Analyzer v2
# =========================
def analyze_trend_per_machine_v2(
    df: pd.DataFrame,
    intent: dict,
    ts_col="ts",
    machine_col="machine_id",
    value_col="value",
    agg="mean",
    z_th=3.0,          # global z-score ambang
    rz_th=3.0,         # rolling z-score ambang
    k_mad=4.5,         # robust MAD ambang
    cv_th=0.10,        # fluktuasi besar jika CV > ini
    roll_win_map=None  # map frekuensi -> window
):
    """
    Analisis per-mesin yang komprehensif:
    - min/max RAW (langsung dari data) + min/max RESAMPLED
    - outlier multi-metode: z-score global, rolling z-score, robust MAD
    - periode fluktuasi (top-k jendela CV tertinggi)
    - tren (arah & %)
    """
    if roll_win_map is None:
        roll_win_map = {"1H": 24, "1D": 7, "1W-MON": 6, "1MS": 3}

    start = intent["time_range"]["start"]
    end   = intent["time_range"]["end"]
    scope = intent["scope"]
    gran  = (intent.get("granularity") or "DAY").upper()
    machines = (intent.get("machines") or [])

    df1 = df.copy()
    df1[ts_col] = pd.to_datetime(df1[ts_col], errors="coerce")
    df1 = df1.dropna(subset=[ts_col, value_col])
    if start: df1 = df1[df1[ts_col] >= pd.Timestamp(start)]
    if end:   df1 = df1[df1[ts_col] <= pd.Timestamp(end)]
    if scope == "SUBSET" and machines:
        df1 = df1[df1[machine_col].isin(machines)]
    if df1.empty:
        return {"empty": True, "reason": "No data in range/selection", "intent": intent}

    pref = {"HOUR":"1H","DAY":"1D","WEEK":"1W-MON","MONTH":"1D","YEAR":"1MS","ALL":"1D"}.get(gran, "1D")
    auto = _auto_freq(start, end)
    order = ["1H","1D","1W-MON","1MS","1Y"]
    freq = pref if order.index(pref) <= order.index(auto) else auto
    rwin = roll_win_map.get(freq, 7)

    results = {}
    for mid, g0 in df1.groupby(machine_col):
        g0 = g0.sort_values(ts_col)
        s_raw = g0.set_index(ts_col)[value_col].astype(float)

        # ---------- min/max RAW (akurasi penuh)
        vmin_raw, vmax_raw = float(s_raw.min()), float(s_raw.max())
        tmin_raw, tmax_raw = s_raw.idxmin().isoformat(), s_raw.idxmax().isoformat()

        # ---------- resample untuk tren & fluktuasi makro
        s = s_raw.resample(freq).agg(agg).dropna()
        if s.empty:
            results[mid] = {"empty": True}
            continue

        # tren
        tstats = _trend_stats(s)

        # min/max RESAMPLED (berguna untuk laporan periodik)
        vmin_res, vmax_res = float(s.min()), float(s.max())
        tmin_res, tmax_res = s.idxmin().isoformat(), s.idxmax().isoformat()

        # volatilitas makro
        std = float(s.std(ddof=0))
        mean = float(s.mean())
        cv = float(std/mean) if mean != 0 else 0.0

        # ---------- OUTLIER multi-metode di RAW (lebih sensitif)
        # z-score global
        mu, sig = s_raw.mean(), s_raw.std(ddof=0)
        sig = sig if sig and sig != 0 else 1.0
        z = (s_raw - mu) / sig

        # robust MAD
        med, madv = _mad(s_raw)
        r = 0.6745 * (s_raw - med) / (madv or 1.0)

        # rolling z-score (deteksi spike lokal)
        rz = _rolling_z(s_raw, win=rwin)

        mask = (z.abs() >= z_th) | (r.abs() >= k_mad) | (rz.abs() >= rz_th)
        outs = []
        if mask.any():
            # urutkan berdasarkan skor terbesar (gabungkan magnitude terbaik dari ketiganya)
            score = np.maximum.reduce([
                z.abs().fillna(0).values,
                r.abs().fillna(0).values,
                (rz.abs().fillna(0).values if isinstance(rz, pd.Series) else np.zeros_like(z.values))
            ])
            order_idx = np.argsort(-score)  # descending
            idx = mask.index.to_list()
            for i in order_idx[:50]:  # batas 50 titik
                ts = idx[i]
                if not mask.loc[ts]:
                    continue
                outs.append({
                    "ts": ts.isoformat(),
                    "value": float(s_raw.loc[ts]),
                    "z": float(z.loc[ts]) if ts in z.index else None,
                    "rz": float(rz.loc[ts]) if (isinstance(rz, pd.Series) and ts in rz.index) else None,
                    "r": float(r.loc[ts]) if ts in r.index else None
                })

        # ---------- Periode fluktuasi (top windows)
        top_win = _top_fluctuation_windows(s, win=max(3, min(rwin, len(s))), k=3)

        # flag fluktuasi besar per-mesin
        flg = (cv > cv_th) or (len(outs) >= 1)

        results[mid] = {
            "points": int(len(s)),
            "trend": tstats,
            "raw_min": {"value": vmin_raw, "ts": tmin_raw},
            "raw_max": {"value": vmax_raw, "ts": tmax_raw},
            "resampled_min": {"value": vmin_res, "ts": tmin_res},
            "resampled_max": {"value": vmax_res, "ts": tmax_res},
            "volatility": {"std": std, "mean": mean, "cv": cv},
            "outliers": {"count": len(outs), "points": outs},
            "fluctuation_windows": top_win,
            "fluctuation": flg,
            "freq_used": freq,
            "roll_window": rwin
        }

    combined = {
        "any_fluctuation": any(v.get("fluctuation") for v in results.values() if not v.get("empty")),
        "machines_with_outlier": [m for m,v in results.items() if (not v.get("empty")) and v["outliers"]["count"]>0],
        "max_cv": max([v["volatility"]["cv"] for v in results.values() if not v.get("empty")], default=0.0)
    }
    return {"intent": intent, "freq": freq, "machines": results, "combined": combined}

# =========================
#  Pretty printer v2
# =========================
def pretty_per_machine_v2(report: dict, decimal_comma=True, places=6, show_outliers=3, show_windows=3):
    if report.get("empty"):
        return f"Tidak ada data ({report.get('reason','')})"
    def fmt(x, p=6):
        if x is None or (isinstance(x,float) and (np.isnan(x) or np.isinf(x))): return "NA"
        s = f"{x:.{p}f}"
        return s.replace(".", ",") if decimal_comma else s
    lines = []
    for mid, r in report["machines"].items():
        if r.get("empty"):
            lines.append(f"[{mid}] kosong.")
            continue
        t = r["trend"]; dir_map = {"up":"meningkat","down":"menurun","flat":"stabil", None:"-"}
        lines.append(
            f"[{mid}] tren {dir_map.get(t['direction'])} ({fmt((t['pct'] or 0)*100, places)}%). "
            f"CV={fmt(r['volatility']['cv'], places)}, std={fmt(r['volatility']['std'], places)} "
            f"(freq={r['freq_used']}, roll_win={r['roll_window']})."
        )
        # min/max RAW (lebih akurat) + resampled
        lines.append(
            f"  RAW  : min={fmt(r['raw_min']['value'], places)} ({r['raw_min']['ts'][:19]}), "
            f"max={fmt(r['raw_max']['value'], places)} ({r['raw_max']['ts'][:19]})."
        )
        lines.append(
            f"  RESMP: min={fmt(r['resampled_min']['value'], places)} ({r['resampled_min']['ts'][:19]}), "
            f"max={fmt(r['resampled_max']['value'], places)} ({r['resampled_max']['ts'][:19]})."
        )
        # outliers detail
        pts = r['outliers']['points'][:show_outliers]
        if pts:
            lines.append(f"  Outliers ({len(r['outliers']['points'])}):")
            for p in pts:
                lines.append(
                    f"    - {p['ts'][:19]} | val={fmt(p['value'], places)} | z={fmt(p['z'],3)} | rz={fmt(p['rz'],3)} | r={fmt(p['r'],3)}"
                )
        else:
            lines.append("  Outliers: tidak terdeteksi.")
        # fluctuation windows
        wins = r["fluctuation_windows"][:show_windows]
        if wins:
            lines.append("  Periode fluktuasi tertinggi:")
            for w in wins:
                lines.append(
                    f"    - {w['start'][:19]} → {w['end'][:19]} | std={fmt(w['std'],places)} | mean={fmt(w['mean'],places)} | cv={fmt(w['cv'],places)}"
                )
    comb = report["combined"]
    lines.append(
        f"(GABUNGAN) ada_fluktuasi={comb['any_fluctuation']}, max_CV={fmt(comb['max_cv'], places)}, "
        f"mesin_dengan_outlier={comb['machines_with_outlier']}"
    )
    return "\n".join(lines)


In [81]:
examples = [
    "Tolong tampilkan ringkasan semua mesin dari november sampai desember tahun lalu ini",
    "Cek tren mesin XP888A hari ini",
    "Apakah ada fluktuasi besar pada mesin XP888A dan PQ667 bulan januari tahun ini?",
    "Mohon tampilkan nilai maksimal semua mesin tahun ini",
    "Tunjukkan tren mesin di bulan April tahun lalu dari awal bulan sampai pertengahan",
]
df_raw=pd.read_csv(r'/workspace/df_raw.csv')
print("\n=== DEMO PREDIKSI ===")
for q in examples:
    try:
        # print("\nQ:", q)
        # pred=runner.predict(q)
        # intent=interpret(q, pred)
        # # subintent=extract_subintent(q)
        # # result=analyze_trend(df_raw, intent, ts_col="ts", machine_col="machine_id", value_col="value")
        # # response = format_trend_response(result, subintent)

        # subintent = extract_subintent(q)   # dari blok sebelumnya
        # report = analyze_trend_per_machine_v2(
        #     df_raw, intent,
        #     ts_col="ts", machine_col="machine_id", value_col="value",
        #     agg="mean",
        #     z_th=3.0, rz_th=3.0, k_mad=4.5,  # sensitivitas outlier
        #     cv_th=0.10                        # ambang fluktuasi besar per mesin
        # )
        # print(pretty_per_machine_v2(report, decimal_comma=True, places=6))
        # q = "Apakah ada fluktuasi besar pada mesin XP888A dan PQ667 bulan januari tahun ini?"
        pred = runner.predict(q)
        intent = interpret(q, pred)
        subintent = extract_subintent(q)
        
        report = analyze_trend_per_machine_v2(
            df_raw, intent,
            ts_col="ts", machine_col="machine_id", value_col="value",
            agg="mean",
            z_th=3.0, rz_th=3.0, k_mad=4.5,  # sensitivitas outlier
            cv_th=0.10                        # ambang fluktuasi besar per mesin
        )
        result = serve_insight_with_charts(
            df_raw, intent,
            report_fn=analyze_trend_per_machine_v2,
            subintent=subintent,
            out_dir="./plots",
            ts_col="ts", machine_col="machine_id", value_col="value"
        )
        
        if result.get("empty"):
            print("No data:", result["reason"])
        else:
            # kirim result["meta"] ke LLM untuk narasi,
            # lalu tampilkan gambar dari result["plot_paths"]
            print("Plot files:", result["plot_paths"])
        
        print(pretty_per_machine_v2(report, decimal_comma=True, places=6))
   
        
    except Exception as e:
        print("Lewati (pastikan model terlatih & path benar):", e)


=== DEMO PREDIKSI ===
No data: No data in range/selection
Tidak ada data (No data in range/selection)
No data: No data in range/selection
Tidak ada data (No data in range/selection)
Plot files: {'PQ667': ['./plots/PQ667_raw.png', './plots/PQ667_resampled_1D.png', './plots/PQ667_hist.png'], 'XP888A': ['./plots/XP888A_raw.png', './plots/XP888A_resampled_1D.png', './plots/XP888A_hist.png']}
[PQ667] tren menurun (-0,069102%). CV=0,002097, std=0,266508 (freq=1D, roll_win=7).
  RAW  : min=90,223010 (2025-01-13T00:00:57), max=149,287100 (2025-01-24T02:00:57).
  RESMP: min=126,719304 (2025-01-03T00:00:00), max=127,918068 (2025-01-01T00:00:00).
  Outliers (50):
    - 2025-01-13T00:03:57 | val=90,223010 | z=-31,020 | rz=-1,342 | r=-140,630
    - 2025-01-13T00:00:57 | val=90,223010 | z=-31,020 | rz=-1,905 | r=-140,630
    - 2025-01-13T00:06:57 | val=95,499737 | z=-26,579 | rz=-0,751 | r=-120,522
  Periode fluktuasi tertinggi:
    - 2025-01-01T00:00:00 → 2025-01-04T00:00:00 | std=0,495823 | mean=

In [79]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
from pathlib import Path

def _ensure_dir(p):
    Path(p).parent.mkdir(parents=True, exist_ok=True)

def _coerce_numeric(x):
    # dukung koma desimal
    if isinstance(x, str):
        x = x.replace(",", ".")
    return pd.to_numeric(x, errors="coerce")

def plot_raw_series(df_raw, machine_id, *,
                    ts_col="ts", val_col="value", machine_col="machine_id",
                    save_path=None, title_suffix="RAW"):
    d = df_raw[df_raw[machine_col] == machine_id].copy()
    if d.empty: return None
    d[ts_col] = pd.to_datetime(d[ts_col], errors="coerce")
    d[val_col] = d[val_col].apply(_coerce_numeric)
    d = d.dropna(subset=[ts_col, val_col]).sort_values(ts_col)
    if d.empty: return None

    plt.figure()
    plt.plot(d[ts_col], d[val_col], marker=".", linestyle="-")
    plt.xlabel("Waktu"); plt.ylabel(val_col); plt.title(f"{machine_id} — {title_suffix}")
    plt.tight_layout()
    if save_path:
        _ensure_dir(save_path); plt.savefig(save_path, dpi=140); plt.close(); return save_path
    return None

def plot_resampled_series(df_raw, machine_id, *,
                          freq="1D", agg="mean",
                          ts_col="ts", val_col="value", machine_col="machine_id",
                          save_path=None, title_suffix="RESAMPLED"):
    d = df_raw[df_raw[machine_col] == machine_id].copy()
    if d.empty: return None

    # pastikan datetime index + numeric value
    d[ts_col] = pd.to_datetime(d[ts_col], errors="coerce")
    d[val_col] = d[val_col].apply(_coerce_numeric)
    d = d.dropna(subset=[ts_col, val_col]).sort_values(ts_col)
    if d.empty: return None

    s = d.set_index(ts_col)[val_col].astype(float)
    # <- inilah kunci: paksa menjadi DatetimeIndex
    s.index = pd.DatetimeIndex(s.index)
    s = s.resample(freq).agg(agg).dropna()
    if s.empty: return None

    plt.figure()
    plt.plot(s.index, s.values, marker="o", linestyle="-")
    plt.xlabel("Waktu"); plt.ylabel(val_col); plt.title(f"{machine_id} — {title_suffix} ({freq})")
    plt.tight_layout()
    if save_path:
        _ensure_dir(save_path); plt.savefig(save_path, dpi=140); plt.close(); return save_path
    return None

# --- di dispatcher, pastikan param 'machine_col' diteruskan ke plotters ---
def serve_insight_with_charts(
    df_raw: pd.DataFrame,
    intent: dict,
    report_fn,            # analyze_trend_per_machine_v2
    subintent: str = "summary",
    out_dir: str = "./plots",
    ts_col="ts", machine_col="machine_id", value_col="value"
):
    si = (subintent or "summary").lower()
    only_minmax = si in {"min_only","max_only","minmax"}

    report = report_fn(df_raw, intent, ts_col=ts_col, machine_col=machine_col, value_col=value_col)
    if report.get("empty"):
        return {"empty": True, "reason": report.get("reason", "no data")}

    meta = package_metadata_for_llm(report, limit_outliers=5)

    if only_minmax:
        return {"meta": meta, "plot_paths": {}, "skipped_plots": True}

    plot_paths = {}
    freq_used = report.get("freq", "1D")
    Path(out_dir).mkdir(parents=True, exist_ok=True)

    for mid in report["machines"].keys():
        p_raw = os.path.join(out_dir, f"{mid}_raw.png")
        p1 = plot_raw_series(df_raw, mid, ts_col=ts_col, val_col=value_col, machine_col=machine_col,
                             save_path=p_raw, title_suffix="RAW")

        p_res = os.path.join(out_dir, f"{mid}_resampled_{freq_used}.png")
        p2 = plot_resampled_series(df_raw, mid, freq=freq_used, agg="mean",
                                   ts_col=ts_col, val_col=value_col, machine_col=machine_col,
                                   save_path=p_res, title_suffix="RESAMPLED")

        p_hist = os.path.join(out_dir, f"{mid}_hist.png")
        # histogram opsional; aman skip bila <2 nilai
        vals = df_raw.loc[df_raw[machine_col]==mid, value_col]
        if not vals.empty:
            vals = vals.apply(_coerce_numeric).dropna()
            if len(vals) >= 2:
                plt.figure()
                plt.hist(vals, bins=30)
                plt.xlabel(value_col); plt.ylabel("Count"); plt.title(f"{mid} — Histogram")
                plt.tight_layout()
                _ensure_dir(p_hist); plt.savefig(p_hist, dpi=140); plt.close()
                p3 = p_hist
            else:
                p3 = None
        else:
            p3 = None

        plot_paths[mid] = [p for p in [p1, p2, p3] if p]

    return {"meta": meta, "plot_paths": plot_paths, "skipped_plots": False}


In [53]:
import pandas as pd

df_raw = pd.read_csv('/workspace/df_raw.csv')  # pastikan path ini benar
df_raw['ts'] = pd.to_datetime(df_raw['ts'], errors='coerce')

print("rows:", len(df_raw))
print("range:", df_raw['ts'].min(), "→", df_raw['ts'].max())
print("machines:", df_raw['machine_id'].nunique(), df_raw['machine_id'].unique()[:10])


rows: 33558
range: 2025-01-01 12:00:57 → 2025-02-05 10:54:57
machines: 2 ['PQ667' 'XP888A']


In [6]:
from data_loader import build_df_raw
from case1_path_pipeline import run_query  # <-- ambil fungsi pipeline final

if __name__ == "__main__":
    # 1. Load df_raw sekali di awal
    excel_path = "/workspace/Machine_Value_Processed.xlsx"
    sheet_name = "SW_3min"
    csv_save   = "/workspace/df_raw.csv"

    df_raw, _ = build_df_raw(
        excel_path=excel_path,
        sheet_name=sheet_name,
        save_csv=csv_save,
        with_lag=False,
    )

    print(f"[INFO] df_raw loaded: {len(df_raw)} rows. CSV saved to {csv_save}")

    # 2. Query contoh (bisa diganti input user realtime)
    examples = [
        "Tolong tampilkan ringkasan semua mesin dari november sampai desember tahun lalu ini",
        "Cek tren mesin XP888A hari ini",
        "Apakah ada fluktuasi besar pada mesin XP888A dan PQ667 bulan januari tahun ini?",
        "Mohon tampilkan nilai maksimal semua mesin tahun ini",
        "Tunjukkan tren mesin di bulan April tahun lalu dari awal bulan sampai pertengahan",
    ]

    for q in examples:
        print("\n=== QUERY:", q)
        result = run_query(q, df_raw, plots_dir="./plots")

        # (a) Laporan bahasa natural untuk operator
        print("\n--- HUMAN REPORT ---")
        print(result["human_report"])

        # (b) Intent terstruktur
        print("\n--- INTENT ---")
        print(result["intent"])

        # (c) Metadata numerik ringkas per mesin
        print("\n--- LLM METADATA ---")
        print(result["llm_metadata"])

        # (d) Lokasi file plot yang baru disimpan
        print("\n--- PLOT PATHS ---")
        print(result["plot_paths"])

    print("\nSelesai.")


[INFO] df_raw loaded: 33558 rows. CSV saved to /workspace/df_raw.csv

=== QUERY: Tolong tampilkan ringkasan semua mesin dari november sampai desember tahun lalu ini

--- HUMAN REPORT ---
Tidak ada data. No data in range/selection

--- INTENT ---
{'query': 'Tolong tampilkan ringkasan semua mesin dari november sampai desember tahun lalu ini', 'scope': 'ALL', 'machines': [], 'granularity': 'MONTH', 'time_range': {'start': '2024-11-01T00:00:00', 'end': '2024-12-31T23:59:59', 'note': 'range bulan'}, 'case': 'RANGE_REL', 'route_hints': ['use_range_extractor', 'slot_month_extractor', 'aggregate_all_machines'], 'meta': {'model_time_label': 'ALL', 'model_time_conf': 0.7819, 'model_case_conf': 0.8879, 'model_scope_conf': 0.9999, 'extractor_kind': 'RANGE'}}

--- LLM METADATA ---
None

--- PLOT PATHS ---
{}

=== QUERY: Cek tren mesin XP888A hari ini

--- HUMAN REPORT ---
Tidak ada data. No data in range/selection

--- INTENT ---
{'query': 'Cek tren mesin XP888A hari ini', 'scope': 'SUBSET', 'machi